# Tutorial 6: Monitoring and Observability with Telemetry

## What You Will Learn

- Understand Scalable's telemetry data model
- Read and analyze JSONL telemetry files
- Generate reports from the Python API
- Build performance analysis from run history
- Configure telemetry options

## Prerequisites

- Completed Tutorial 1
- `pip install scalable`

In [ ]:
import os
import tempfile
import time
import json

project_dir = tempfile.mkdtemp(prefix="scalable-telemetry-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Step 1: Generate Telemetry Data

Let's run a workflow to produce telemetry records.

In [ ]:
from scalable import ScalableSession

manifest = """\
version: 1
project:
  name: telemetry-demo
targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none
components:
  worker:
    cpus: 1
    memory: 1G
tasks:
  compute:
    component: worker
    cache: true
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest)

print("Manifest written.")

In [ ]:
def compute_task(n: int) -> dict:
    """A task that takes variable time."""
    time.sleep(0.2 + (n % 3) * 0.1)  # Variable duration
    return {"n": n, "result": n ** 2}


# Run a workflow
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()

futures = [client.submit(compute_task, i, tag="worker") for i in range(10)]
results = client.gather(futures)

session.close()
print(f"Workflow complete: {len(results)} tasks")

## Step 2: Telemetry Directory Structure

Every run produces structured JSONL files under `.scalable/runs/`.

In [ ]:
from pathlib import Path

runs_dir = Path(".scalable/runs")

if runs_dir.exists():
    run_dirs = sorted(runs_dir.iterdir())
    latest_run = run_dirs[-1]
    
    print(f"Run directory: {latest_run.name}")
    print(f"\nFiles:")
    for f in sorted(latest_run.iterdir()):
        size = f.stat().st_size
        print(f"  {f.name:20s} {size:>6} bytes")
else:
    print("No telemetry found (check SCALABLE_TELEMETRY=1)")

## Step 3: Read Run Metadata

In [ ]:
run_json = latest_run / "run.json"

if run_json.exists():
    with open(run_json) as f:
        meta = json.load(f)
    
    print("Run Metadata:")
    print(f"  Run ID: {meta.get('run_id', 'N/A')}")
    print(f"  Project: {meta.get('project_name', 'N/A')}")
    print(f"  Target: {meta.get('target_name', 'N/A')}")
    print(f"  Provider: {meta.get('provider_name', 'N/A')}")
    print(f"  Status: {meta.get('status', 'N/A')}")
    print(f"  Started: {meta.get('started_at', 'N/A')}")
    print(f"  Manifest lock: {meta.get('manifest_lock', 'N/A')[:20]}...")

## Step 4: Analyze Task Events

In [ ]:
import pandas as pd

tasks_file = latest_run / "tasks.jsonl"

if tasks_file.exists() and tasks_file.stat().st_size > 0:
    tasks = []
    with open(tasks_file) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))
    
    df = pd.DataFrame(tasks)
    print(f"Task events: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nStates: {df['state'].value_counts().to_dict() if 'state' in df.columns else 'N/A'}")
    
    if 'duration_s' in df.columns:
        completed = df[df['state'] == 'succeeded']
        if not completed.empty:
            print(f"\nDuration statistics:")
            print(f"  Mean: {completed['duration_s'].mean():.3f}s")
            print(f"  Max: {completed['duration_s'].max():.3f}s")
            print(f"  Min: {completed['duration_s'].min():.3f}s")
else:
    print("No task events recorded.")

## Step 5: Worker Events

In [ ]:
workers_file = latest_run / "workers.jsonl"

if workers_file.exists() and workers_file.stat().st_size > 0:
    workers = []
    with open(workers_file) as f:
        for line in f:
            if line.strip():
                workers.append(json.loads(line))
    
    print(f"Worker events: {len(workers)}")
    for w in workers[:5]:
        print(f"  {w}")
else:
    print("No worker events recorded.")

## Step 6: Resource Utilization

In [ ]:
resources_file = latest_run / "resources.jsonl"

if resources_file.exists() and resources_file.stat().st_size > 0:
    resources = []
    with open(resources_file) as f:
        for line in f:
            if line.strip():
                resources.append(json.loads(line))
    
    res_df = pd.DataFrame(resources)
    print(f"Resource events: {len(res_df)}")
    print(f"Columns: {list(res_df.columns)}")
    
    if 'cpu_percent' in res_df.columns:
        print(f"\nCPU utilization: {res_df['cpu_percent'].mean():.1f}% mean")
    if 'memory_mb' in res_df.columns:
        print(f"Memory usage: {res_df['memory_mb'].mean():.0f} MB mean")
else:
    print("No resource events recorded.")

## Step 7: Using the Telemetry Collectors API

In [ ]:
from scalable.telemetry.collectors import iter_run_dirs, read_jsonl

# Iterate all run directories
print("All runs:")
for run_dir in iter_run_dirs(runs_dir):
    run_json_path = run_dir / "run.json"
    if run_json_path.exists():
        with open(run_json_path) as f:
            meta = json.load(f)
        print(f"  {meta.get('run_id', run_dir.name)}: {meta.get('status', '?')}")

# Read task events using helper
task_records = read_jsonl(latest_run / "tasks.jsonl")
print(f"\nTask records via read_jsonl: {len(task_records)}")

## Step 8: Multiple Runs for Trend Analysis

Let's run the workflow a few more times to see trends.

In [ ]:
# Run 2 more times with slightly different workloads
for run_num in range(2):
    sess = ScalableSession.from_yaml("./scalable.yaml", target="local")
    cl = sess.start()
    
    n_tasks = 5 + run_num * 3
    futs = [cl.submit(compute_task, i, tag="worker") for i in range(n_tasks)]
    cl.gather(futs)
    sess.close()
    
    print(f"Run {run_num + 2} complete ({n_tasks} tasks)")
    time.sleep(0.5)

print(f"\nTotal runs: {len(list(iter_run_dirs(runs_dir)))}")

In [ ]:
# Analyze trends across runs
summaries = []

for run_dir in iter_run_dirs(runs_dir):
    run_json_path = run_dir / "run.json"
    if not run_json_path.exists():
        continue
    
    with open(run_json_path) as f:
        meta = json.load(f)
    
    task_records = read_jsonl(run_dir / "tasks.jsonl")
    succeeded = [t for t in task_records if t.get("state") == "succeeded"]
    
    summaries.append({
        "run_id": meta.get("run_id", "")[:30],
        "status": meta.get("status"),
        "tasks": len(succeeded),
        "mean_duration": (
            sum(t.get("duration_s", 0) for t in succeeded) / len(succeeded)
            if succeeded else 0
        ),
    })

history_df = pd.DataFrame(summaries)
print("Run History:")
print(history_df.to_string(index=False))

## Step 9: Telemetry Configuration

In [ ]:
from scalable.common import settings

print("Telemetry settings:")
print(f"  Enabled: {settings.telemetry_enabled} (SCALABLE_TELEMETRY)")
print(f"  Parquet export: {settings.telemetry_parquet} (SCALABLE_TELEMETRY_PARQUET)")
print(f"  Runs directory: {settings.runs_dir} (SCALABLE_RUNS_DIR)")
print(f"\nTo disable: export SCALABLE_TELEMETRY=0")
print(f"To enable Parquet: export SCALABLE_TELEMETRY_PARQUET=1")

## Summary

1. Every manifest-driven run records JSONL telemetry automatically
2. `run.json` contains metadata (ID, target, status, timestamps)
3. `tasks.jsonl` records full task lifecycle with durations
4. `resources.jsonl` tracks CPU/memory utilization
5. `workers.jsonl` records worker lifecycle events
6. `iter_run_dirs` and `read_jsonl` provide programmatic access
7. Historical analysis reveals performance trends

## Next Steps

- **Tutorial 7**: Use failure events for error diagnosis
- **Tutorial 9**: Feed telemetry to ML advisor for predictions
- **Tutorial 5**: Monitor cloud costs through telemetry

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")